In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import json
import os
import shutil

from sklearn.model_selection import train_test_split

In [2]:
main_path = "/kaggle/input/wlasl-processed/"
output_folder = '/kaggle/working/'
wlasl_df = pd.read_json(main_path + "WLASL_v0.3.json")

In [4]:
wlasl_df.head(5)

,gloss,instances
0,book,"[{'bbox': [385, 37, 885, 720], 'fps': 25, 'fra..."
1,drink,"[{'bbox': [551, 68, 1350, 1080], 'fps': 25, 'f..."
2,computer,"[{'bbox': [0, 0, 360, 240], 'fps': 25, 'frame_..."
3,before,"[{'bbox': [0, 0, 360, 240], 'fps': 25, 'frame_..."
4,chair,"[{'bbox': [0, 0, 360, 240], 'fps': 25, 'frame_..."


In [5]:
def get_videos_ids(json_list):
    """
    check if the video id is available in the dataset
    and return the viedos ids of the current instance
    
    Args:
        json_list: Instance of video metadata.
        
    Returns:
        List of video ids.
    """
    video_ids = []
    for ins in json_list:
        video_id = ins['video_id']
        if os.path.exists(f'{main_path}videos/{video_id}.mp4'):
            video_ids.append(video_id)
    return video_ids

In [6]:
def get_json_features(json_list):
    """
    function to check if the video id is available in the dataset
    and return the viedos ids and url or any other featrue of the current instance
    
    input: instance json list
    output: list of videos_ids
    
    """
    videos_ids = []
    videos_urls = []
    videos_bbox = []
    videos_fps = []
    videos_frame_end = []
    videos_frame_start = []
    videos_signer_id = []
    videos_source = []
    videos_split = []
    videos_variation_id = []
    for ins in json_list:
      
        video_id = ins['video_id']
        video_url = ins['url']
        video_bbox = ins['bbox']
        video_fps = ins['fps']
        video_frame_end = ins['frame_end']
        video_frame_start = ins['frame_start']
        video_signer_id = ins['signer_id']
        video_source = ins['source']
        video_split = ins['split']
        video_variation_id = ins['variation_id']
        if os.path.exists(f'{main_path}videos/{video_id}.mp4'):
            videos_ids.append(video_id)
            videos_urls.append(video_url)
            videos_bbox.append(video_bbox)
            videos_fps.append(video_fps)
            videos_frame_end.append(video_frame_end)
            videos_frame_start.append(video_frame_start)
            videos_signer_id.append(video_signer_id)
            videos_source.append(video_source)
            videos_split.append(video_split)
            videos_variation_id.append(video_variation_id)
    return videos_ids, videos_urls, videos_bbox, videos_fps, videos_frame_end, videos_frame_start, videos_signer_id, videos_source, videos_split,videos_variation_id

In [7]:
with open(main_path+'WLASL_v0.3.json', 'r') as data_file:
    json_data = data_file.read()

instance_json = json.loads(json_data)

In [8]:
wlasl_df["video_ids"] = wlasl_df["instances"].apply(get_videos_ids)

In [9]:
wlasl_df.head(5)

,gloss,instances,video_ids
0,book,"[{'bbox': [385, 37, 885, 720], 'fps': 25, 'fra...","[69241, 07069, 07068, 07070, 07099, 07074]"
1,drink,"[{'bbox': [551, 68, 1350, 1080], 'fps': 25, 'f...","[69302, 65539, 17710, 17733, 65540, 17734, 177..."
2,computer,"[{'bbox': [0, 0, 360, 240], 'fps': 25, 'frame_...","[12328, 12312, 12311, 12338, 12313, 12314, 123..."
3,before,"[{'bbox': [0, 0, 360, 240], 'fps': 25, 'frame_...","[05728, 05749, 05750, 05729, 05730, 65167, 057..."
4,chair,"[{'bbox': [0, 0, 360, 240], 'fps': 25, 'frame_...","[09848, 09869, 09849, 09850, 09851, 65328, 09854]"


In [14]:
features_df = pd.DataFrame(columns=['gloss', 'video_id', 'urls', 'bbox', 'fps', 'frame_end', 'frame_start','signer_id', 'source', 'split', 'variation_id'])
for row in wlasl_df.iterrows():
    ids, urls, bbox, fps, frame_end, frame_start,signer_id, source, split, variation_id = get_json_features(row[1][1])
    word = [row[1][0]] * len(ids)
    df = pd.DataFrame(list(zip(word, ids, urls, bbox, fps, frame_end, frame_start, signer_id, source, split, variation_id)), columns=features_df.columns)
    features_df = features_df.append(df, ignore_index=True)


In [15]:
features_df.head()

,gloss,video_id,urls,bbox,fps,frame_end,frame_start,signer_id,source,split,variation_id
0,book,69241,http://aslbricks.org/New/ASL-Videos/book.mp4,"[385, 37, 885, 720]",25,-1,1,118,aslbrick,train,0
1,book,07069,https://signstock.blob.core.windows.net/signsc...,"[462, 44, 949, 720]",25,-1,1,31,signschool,train,0
2,book,07068,https://s3-us-west-1.amazonaws.com/files.start...,"[234, 17, 524, 414]",25,-1,1,36,startasl,train,0
3,book,07070,https://media.asldeafined.com/vocabulary/14666...,"[131, 26, 526, 480]",25,-1,1,59,asldeafined,train,0
4,book,07099,http://www.aslsearch.com/signs/videos/book.mp4,"[162, 54, 528, 400]",25,-1,1,12,aslsearch,val,0


In [28]:
len(features_df.gloss.unique())

2000

In [20]:
train_mask = features_df['split'] == 'train'
val_mask = features_df['split'] == 'val'
test_mask = features_df['split'] == 'test'

train_pos = np.flatnonzero(train_mask)
val_pos =  np.flatnonzero(val_mask)
test_pos = np.flatnonzero(test_mask)

train = features_df.iloc[train_pos]

val = features_df.iloc[val_pos]

test = features_df.iloc[test_pos]


In [24]:
x_train = train.loc[:, train.columns != 'gloss']
y_train = train['gloss']

x_val = val.loc[:, val.columns != 'gloss']
y_val = val['gloss']

x_test = test.loc[:, test.columns != 'gloss']
y_test = test['gloss']

In [26]:
print(len(x_train),len(x_test), len(x_val))

8313 1414 2253


In [ ]:
x_train[0]

In [ ]:
import os

In [ ]:
import shutil
shutil.rmtree("./data")

In [ ]:
!pip install nqdm
from nqdm import nqdm

In [ ]:
from distutils.dir_util import copy_tree


def generateDatasplitFolder(series, folderName):
    new_path = output_folder+'data/'+str(folderName)
    if not os.path.exists(new_path):
        os.makedirs(new_path)
        for val in series:
            from_directory = main_path+'videos/'+str(val)+'.mp4'
            to_directory = output_folder+'data/'+str(val)+'.mp4'
            shutil.copy(from_directory, to_directory)
        
        

    
    
    

In [ ]:
generateDatasplitFolder(x_train, 'train')

In [ ]:
generateDatasplitFolder(x_val, 'val')

In [ ]:
generateDatasplitFolder(x_test, 'test')

In [ ]:
!cd /kaggle/working

In [ ]:
!tar -czf WLASL_DataSplit.tar.gz data

from IPython.display import FileLink

FileLink(r'WLASL_DataSplit.tar.gz')